<a href="https://colab.research.google.com/github/marsanla/C4L/blob/main/SOL_C4L_6_Ejercicio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Soluciones — Módulo 6: Proyecto final integrador

> Solución completa del **sistema de alertas y reporting** para un despacho de abogados. Sigue paso a paso el enunciado del notebook C4L_6 y aplica todo lo aprendido: funciones, archivos, fechas, pandas, matplotlib, JSON, manejo de errores, envío de emails (SMTP) e informes PDF con `fpdf`.

### Estructura (siguiendo el enunciado del C4L_6)

1. **Carga y análisis de datos** — leer CSV, calcular días restantes y representar gráficamente la urgencia por tipo de caso y abogado.
2. **Sistema de alertas** — detectar plazos próximos (≤ 7 días) y enviar emails personalizados (texto + HTML) a los abogados responsables.
3. **Generación de informes** — exportar a CSV y a PDF con `fpdf`, y enviarlos por email al departamento de reporting.
4. **Visualización** — gráficos de casos a tiempo vs vencidos vs pendientes, y rendimiento por abogado.

### Cómo está organizada la solución

| Función | Responsabilidad |
| --- | --- |
| `cargar_datos` | Lee el CSV y ajusta tipos de fecha. |
| `calcular_fechas_limite` | Añade columnas con días restantes y ordena. |
| `analizar_fechas` | Genera la gráfica comparativa por tipo y abogado. |
| `enviar_email` | Envía un email (texto plano + HTML + adjuntos) vía SMTP (Mailtrap). |
| `revisar_fechas_y_enviar_alertas` | Detecta vencimientos próximos y envía alertas. |
| `generar_informe_csv` | Exporta el DataFrame a CSV. |
| `generar_informe_pdf` | Exporta el DataFrame a PDF (con `fpdf`). |
| `generar_y_enviar_informe_pendientes` | Combina CSV+PDF y los envía por email. |
| `grafico_casos_presentados_vs_vencidos_y_pendientes` | Gráfico agregado. |
| `grafico_rendimiento_abogados` | Gráfico por abogado. |

> **Importante sobre emails:** para que el envío funcione necesitas una cuenta de [Mailtrap](https://mailtrap.io/) (o cualquier SMTP) y reemplazar `user`/`key` en `enviar_email` por las credenciales del *inbox*. Mientras tanto la variable `DRY_RUN = True` evita contactar el servidor; cámbiala a `False` para enviar de verdad.

### Preparación — generar datos de ejemplo


In [ ]:
import pandas as pd
import re
import random
from datetime import datetime, timedelta

random.seed(42)   # reproducibilidad


def to_snake_case(text):
    # Convertir el texto a minúsculas
    text = text.lower()
    # Reemplazar los espacios y otros caracteres especiales con un guion bajo
    text = re.sub(r'[^a-z0-9]+', '_', text)
    # Eliminar los guiones bajos al principio y al final
    return text.strip('_')


def generar_datos_casos(num_casos=60):
    """Genera un DataFrame con datos simulados para casos legales.

    Sigue el enunciado del C4L_6: 5 abogados, 5 tipos de caso, 3 estados,
    columnas de email y fecha de finalización, y caso_id con formato C-XXXX.
    """
    nombres    = ['Juan Perez', 'Ana Gomez', 'Carlos Ruiz', 'Laura Martínez', 'Roberto Díaz']
    tipos_caso = ['Civil', 'Penal', 'Comercial', 'Laboral', 'Familia']
    estados    = ['Ganado', 'Perdido', 'Pendiente']

    datos = {
        'caso_id': [], 'abogado': [], 'email': [],
        'fecha_inicio': [], 'fecha_finalizacion': [],
        'valor_disputado': [], 'horas_dedicadas': [],
        'tipo_caso': [], 'estado': [],
        'fecha_limite_fase1': [], 'fecha_limite_fase2': [],
    }

    # Fecha base relativa al "hoy" del notebook para que la demo siempre tenga
    # casos con fases en el futuro y otros vencidos.
    fecha_inicio_base = datetime.now() - timedelta(days=200)

    for i in range(1, num_casos + 1):
        abogado          = random.choice(nombres)
        fecha_inicio     = fecha_inicio_base + timedelta(days=random.randint(0, 365))
        valor_disputado  = random.randint(5, 100) * 1000   # 5.000 € – 100.000 €
        horas_dedicadas  = random.randint(5, 40)
        tipo_caso        = random.choice(tipos_caso)
        estado           = random.choice(estados)
        fase1_delta      = timedelta(days=random.randint(30, 90))
        fase2_delta      = timedelta(days=random.randint(120, 180))
        caso_id          = f"C-{i:04d}"

        fecha_limite_fase1 = fecha_inicio + fase1_delta
        fecha_limite_fase2 = fecha_inicio + fase2_delta

        # Solo los casos no pendientes tienen fecha de finalización
        fecha_finalizacion = None
        if estado != 'Pendiente':
            rango_dias = (fecha_limite_fase2 + timedelta(days=10) - fecha_inicio).days
            fecha_finalizacion = fecha_inicio + timedelta(days=random.randint(0, rango_dias))

        datos['caso_id'].append(caso_id)
        datos['abogado'].append(abogado)
        datos['email'].append(f"{to_snake_case(abogado)}@mycoollawfirm.com")
        datos['fecha_inicio'].append(fecha_inicio.strftime('%Y-%m-%d'))
        datos['fecha_finalizacion'].append(fecha_finalizacion.strftime('%Y-%m-%d') if fecha_finalizacion else None)
        datos['valor_disputado'].append(valor_disputado)
        datos['horas_dedicadas'].append(horas_dedicadas)
        datos['tipo_caso'].append(tipo_caso)
        datos['estado'].append(estado)
        datos['fecha_limite_fase1'].append(fecha_limite_fase1.strftime('%Y-%m-%d'))
        datos['fecha_limite_fase2'].append(fecha_limite_fase2.strftime('%Y-%m-%d'))

    return pd.DataFrame(datos)


df = generar_datos_casos()
df.to_csv("casos.csv", index=False)
print(f"Generado 'casos.csv' con {len(df)} registros.")
df.head()

## 1. Carga y análisis de datos


In [2]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt


def cargar_datos(archivo_csv):
    """Carga el CSV y convierte las columnas de fecha a datetime."""
    df = pd.read_csv(archivo_csv)
    columnas_fecha = ["fecha_inicio", "fecha_limite_fase1", "fecha_limite_fase2"]
    for col in columnas_fecha:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


datos = cargar_datos("casos.csv")
print("Tipos:")
print(datos.dtypes)
datos.head()

Tipos:
caso_id                        int64
abogado                       object
tipo_caso                     object
valor_disputado                int64
horas_dedicadas                int64
fecha_inicio          datetime64[ns]
fecha_limite_fase1    datetime64[ns]
fecha_limite_fase2    datetime64[ns]
estado                        object
dtype: object


,caso_id,abogado,tipo_caso,valor_disputado,horas_dedicadas,fecha_inicio,fecha_limite_fase1,fecha_limite_fase2,estado
0,1000,Carlos Ruiz,Penal,63513,40,2025-05-22,2025-06-18,2025-07-09,Pendiente
1,1001,Laura Martin,Civil,12811,28,2025-07-09,2025-08-03,2025-09-29,Pendiente
2,1002,Juan Perez,Familia,57124,188,2025-12-16,2026-02-06,2026-04-05,Resuelto
3,1003,Carlos Ruiz,Civil,46853,183,2025-12-23,2026-02-09,2026-04-07,Resuelto
4,1004,Ana Gomez,Comercial,31793,28,2025-10-22,2025-11-28,2025-12-27,Resuelto


In [3]:
def calcular_fechas_limite(datos):
    """Añade columnas con los días restantes hasta cada fase."""
    ahora = datetime.now()
    datos = datos.copy()
    datos["dias_restantes_fase1"] = (datos["fecha_limite_fase1"] - ahora).dt.days
    datos["dias_restantes_fase2"] = (datos["fecha_limite_fase2"] - ahora).dt.days

    # Ordenar por urgencia (la fase 1 manda)
    datos = datos.sort_values(["dias_restantes_fase1", "dias_restantes_fase2"])
    return datos


datos_analizados = calcular_fechas_limite(datos)
datos_analizados.head()

,caso_id,abogado,tipo_caso,valor_disputado,horas_dedicadas,fecha_inicio,fecha_limite_fase1,fecha_limite_fase2,estado,dias_restantes_fase1,dias_restantes_fase2
29,1029,Laura Martin,Comercial,33644,68,2025-05-15,2025-06-10,2025-07-03,Pendiente,-339,-316
0,1000,Carlos Ruiz,Penal,63513,40,2025-05-22,2025-06-18,2025-07-09,Pendiente,-331,-310
38,1038,Carlos Ruiz,Comercial,58545,176,2025-06-01,2025-06-26,2025-08-11,Resuelto,-323,-277
50,1050,Ana Gomez,Penal,113080,11,2025-06-01,2025-06-30,2025-08-04,Pendiente,-319,-284
18,1018,Ana Gomez,Penal,103019,200,2025-05-29,2025-07-07,2025-09-05,Pendiente,-312,-252


In [ ]:
# Genera una gráfica que muestra los días restantes comparados entre sí,
# y categorizados por tipo de caso y abogado.
def analizar_fechas(datos):
    """Scatter: días restantes fase1 vs fase2, color por tipo y marcador por abogado."""
    # Mapa de colores para los tipos de caso (mismos colores que el enunciado)
    color_map = {
        'Civil':     'blue',
        'Penal':     'red',
        'Comercial': 'green',
        'Laboral':   'purple',
        'Familia':   'orange',
    }
    # Mapa de marcadores para los abogados (5 abogados, como el enunciado)
    marker_map = {
        'Juan Perez':     'o',  # Círculo
        'Ana Gomez':      's',  # Cuadrado
        'Carlos Ruiz':    '^',  # Triángulo
        'Laura Martínez': 'p',  # Pentágono
        'Roberto Díaz':   'D',  # Diamante
    }

    # Crear una nueva figura para la gráfica
    plt.figure(figsize=(11, 6))

    # Agrupar los datos por tipo de caso y abogado, y generar un scatter plot para cada grupo
    for (tipo_caso, abogado), group in datos.groupby(['tipo_caso', 'abogado']):
        plt.scatter(
            group["dias_restantes_fase1"],
            group["dias_restantes_fase2"],
            c=color_map.get(tipo_caso, "gray"),
            marker=marker_map.get(abogado, "x"),
            s=80,
            edgecolors="black",
            alpha=0.7,
            label=f"{tipo_caso} — {abogado}",
        )

    # Líneas de referencia (0 días = ya vencido)
    plt.axhline(0, color="red", linestyle="--", linewidth=0.7)
    plt.axvline(0, color="red", linestyle="--", linewidth=0.7)

    # Configuración del título y etiquetas de los ejes
    plt.title("Días restantes — Fase 1 vs Fase 2")
    plt.xlabel("Días restantes Fase 1")
    plt.ylabel("Días restantes Fase 2")
    # Mostrar la cuadrícula
    plt.grid(True, alpha=0.3)
    # Mostrar la leyenda fuera de la gráfica
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    # Ajustar el layout para evitar recortes
    plt.tight_layout()
    # Mostrar la gráfica
    plt.show()


# Solo casos pendientes
datos_pendientes = datos_analizados[datos_analizados["estado"] == "Pendiente"].copy()
analizar_fechas(datos_pendientes)

## 2. Sistema de alertas

Construimos un mecanismo automatizado que monitoriza los plazos de los casos legales:

* **Seguimiento de fechas límite:** recorremos cada caso y comprobamos si la fase 1 o la fase 2 vence en los próximos **7 días**.
* **Notificación de alertas:** generamos un mensaje detallado (tipo de caso, abogado responsable, días restantes, valor en disputa).
* **Envío de alertas por email:** enviamos el mensaje al abogado responsable en formato **texto plano + HTML** con `smtplib`.

> Para enviar emails reales necesitas un servidor SMTP. Usamos [Mailtrap](https://mailtrap.io/) como sandbox de pruebas — crea una cuenta gratis y sustituye `"user"` y `"key"` en `enviar_email` por las credenciales del *inbox*. Mientras tanto, mantén `DRY_RUN = True` para que la función no contacte el servidor.

In [ ]:
# Importar las librerias necesarias para el envío de emails
# import pandas as pd          --> ya importada
# from datetime import datetime --> ya importada
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
import os


# Si DRY_RUN es True NO se contacta con el servidor SMTP — útil mientras no
# tengas credenciales de Mailtrap configuradas. Cambia a False para enviar.
DRY_RUN = True


def enviar_email(emisor, destinatario, asunto, mensaje, mensaje_html, archivos_adjuntos=None):
    """Envía un email con texto plano + HTML y adjuntos opcionales."""
    # Crear un objeto MIMEMultipart para el mensaje
    message = MIMEMultipart()
    message['From']    = emisor
    message['To']      = destinatario
    message['Subject'] = asunto

    # Adjuntar el mensaje en formato de texto plano y HTML
    message.attach(MIMEText(mensaje, 'plain', 'utf-8'))
    message.attach(MIMEText(mensaje_html, 'html', 'utf-8'))

    # Adjuntar archivos si se proporcionan
    if archivos_adjuntos:
        for archivo in archivos_adjuntos:
            if os.path.isfile(archivo):
                with open(archivo, 'rb') as adjunto:
                    mime_base = MIMEBase('application', 'octet-stream')
                    mime_base.set_payload(adjunto.read())
                    encoders.encode_base64(mime_base)
                    mime_base.add_header(
                        'Content-Disposition',
                        f'attachment; filename={os.path.basename(archivo)}'
                    )
                    message.attach(mime_base)

    if DRY_RUN:
        # Modo simulación — no contactamos con el servidor
        print(f"[DRY-RUN] Email a {destinatario} | Asunto: {asunto}")
        return

    # Configurar el servidor SMTP y enviar el email (Mailtrap sandbox)
    with smtplib.SMTP("sandbox.smtp.mailtrap.io", 2525) as server:
        server.starttls()                                    # Iniciar TLS
        server.login("user", "key")                          # Credenciales Mailtrap
        server.sendmail(emisor, destinatario, message.as_string())
        print(f"Envio de email a: {destinatario}")

In [ ]:
def revisar_fechas_y_enviar_alertas(datos, emisor, dias_aviso=7):
    """Genera y envía alertas para casos cuya fase1 o fase2 vence en <= dias_aviso días."""
    alertas_generadas = 0

    for _, caso in datos.iterrows():
        d1 = caso["dias_restantes_fase1"]
        d2 = caso["dias_restantes_fase2"]

        # Comprobar si alguna fecha límite está próxima (entre 0 y dias_aviso días)
        if (0 <= d1 <= dias_aviso) or (0 <= d2 <= dias_aviso):

            # Mensaje en formato de texto plano
            mensaje = (
                f"AVISO — Caso {caso['caso_id']} ({caso['tipo_caso']})\n"
                f"Abogado responsable: {caso['abogado']}\n"
                f"Días restantes Fase 1: {d1}\n"
                f"Días restantes Fase 2: {d2}\n"
                f"Valor en disputa: {caso['valor_disputado']:,}€\n"
            )

            # Mensaje en formato HTML
            mensaje_html = f"""
            <html>
              <body style="font-family:Arial,sans-serif;">
                <h2 style="color:#b00;">🚨 Alerta de Caso Inminente</h2>
                <p>Hola <b>{caso['abogado']}</b>,</p>
                <p>El caso <b>{caso['caso_id']}</b> ({caso['tipo_caso']}) tiene un plazo próximo.</p>
                <ul>
                  <li>Días restantes Fase 1: <b>{d1}</b></li>
                  <li>Días restantes Fase 2: <b>{d2}</b></li>
                  <li>Valor en disputa: <b>{caso['valor_disputado']:,}€</b></li>
                </ul>
                <p>Revísalo cuanto antes.</p>
              </body>
            </html>
            """

            # Enviar el email con ambos formatos de mensaje
            enviar_email(
                emisor,
                caso['email'],
                "🚨 Alerta de Caso Inminente",
                mensaje,
                mensaje_html,
            )
            alertas_generadas += 1

    print(f"Total alertas generadas: {alertas_generadas}")
    return alertas_generadas


# Dirección de email del emisor
emisor = "info@mycoollawfirm.com"

# Revisar las fechas y enviar alertas basadas en los datos proporcionados
revisar_fechas_y_enviar_alertas(datos_pendientes, emisor)

## 3. Generación de informes

Generamos informes estructurados con el estado actual de los casos legales:

* **Creación de informes detallados** — CSV (siempre) y PDF (con la librería `fpdf`).
* **Distribución por email** — adjuntamos ambos archivos y los enviamos al departamento de reporting con `enviar_email`.

In [ ]:
# Instalar la librería fpdf (si no la tienes instalada)
# !pip install fpdf

from fpdf import FPDF

In [ ]:
# Función para generar informe CSV
def generar_informe_csv(datos, archivo_csv):
    """Guarda el DataFrame como un archivo CSV."""
    # Guardar el DataFrame como un archivo CSV
    datos.to_csv(archivo_csv, index=False)
    # Imprimir confirmación de que el informe CSV ha sido generado
    print(f"Informe CSV guardado en: {archivo_csv}")


generar_informe_csv(datos_pendientes, "informe_casos.csv")


In [ ]:
# Función para generar informe PDF (con la libreria fpdf)
def generar_informe_pdf(datos, archivo_pdf):
    """Genera un PDF con un listado de casos."""
    # Crear un objeto FPDF para el PDF
    pdf = FPDF()
    # Añadir una página al PDF
    pdf.add_page()
    # Establecer la fuente del PDF
    pdf.set_font("Arial", size=11)

    # Añadir un título al PDF
    pdf.set_font("Arial", style="B", size=14)
    pdf.cell(0, 10, txt=f"Informe de casos ({datetime.now().strftime('%d/%m/%Y')})", ln=True)
    pdf.ln(4)
    pdf.set_font("Arial", size=10)

    # Iterar sobre cada fila del DataFrame
    for _, row in datos.iterrows():
        # Añadir una línea al PDF con la información del caso
        linea = (
            f"{row['caso_id']} | {row['tipo_caso']:<10} | {row['abogado']:<18} | "
            f"Estado: {row['estado']:<10} | Fase1: {row['dias_restantes_fase1']:>4}d | "
            f"Fase2: {row['dias_restantes_fase2']:>4}d | "
            f"Valor: {row['valor_disputado']:>7}EUR"
        )
        # fpdf clásico no soporta UTF-8; sustituimos caracteres problemáticos
        pdf.cell(0, 6, txt=linea.encode("latin-1", "replace").decode("latin-1"), ln=True)

    # Guardar el PDF en el archivo especificado
    pdf.output(archivo_pdf)
    # Imprimir confirmación de que el informe PDF ha sido generado
    print(f"Informe PDF guardado en: {archivo_pdf}")


generar_informe_pdf(datos_pendientes, "informe_casos.pdf")

In [ ]:
# Función para generar y enviar informe de casos pendientes
def generar_y_enviar_informe_pendientes(datos, emisor, receptor):
    """Genera CSV+PDF del mes actual y los envía por email al receptor."""
    # Obtener el mes y año actuales
    ahora = datetime.now()
    mes, anio = ahora.month, ahora.year

    # Generar los nombres de archivo base usando el mes y año actuales
    informe_csv = f"informe_pendientes_{anio}_{mes:02d}.csv"
    informe_pdf = f"informe_pendientes_{anio}_{mes:02d}.pdf"

    # Generar y guardar el informe CSV y PDF
    generar_informe_csv(datos, informe_csv)
    generar_informe_pdf(datos, informe_pdf)

    # Mensaje en formato de texto plano
    mensaje = (
        f"Adjunto el reporte mensual de casos pendientes ({mes:02d}/{anio}).\n"
        f"Total de casos pendientes: {len(datos)}.\n"
    )

    # Mensaje en formato HTML
    mensaje_html = f"""
    <html>
      <body style="font-family:Arial,sans-serif;">
        <h2>📄 Reporte mensual de casos pendientes</h2>
        <p>Adjuntamos el informe correspondiente a <b>{mes:02d}/{anio}</b>.</p>
        <p>Total de casos pendientes: <b>{len(datos)}</b>.</p>
      </body>
    </html>
    """

    # Enviar el email con ambos formatos de mensaje y los archivos adjuntos
    enviar_email(
        emisor,
        receptor,
        "📄 Reporte mensual de casos",
        mensaje,
        mensaje_html,
        [informe_csv, informe_pdf],
    )


# Dirección de email del emisor y del receptor
emisor   = "info@mycoollawfirm.com"
receptor = "reporting@mycoollawfirm.com"

# Generar y enviar el informe de casos pendientes
generar_y_enviar_informe_pendientes(datos_pendientes, emisor, receptor)

## 4. Visualización


In [ ]:
# Función para generar la gráfica de casos presentados a tiempo vs. vencidos y pendientes
def grafico_casos_presentados_vs_vencidos_y_pendientes(datos):
    """Gráfico de barras: a tiempo (Ganado/Perdido antes de fase2),
    vencidos (Pendiente con fase2 ya pasada) y pendientes (Pendiente con fase2 futura)."""
    # Convertir las columnas de fechas a tipo datetime para facilitar las comparaciones
    datos = datos.copy()
    datos["fecha_limite_fase2"] = pd.to_datetime(datos["fecha_limite_fase2"], errors="coerce")
    ahora = datetime.now()

    # Filtrar los datos para obtener los casos presentados a tiempo
    #   (estado distinto de "Pendiente" -> el caso ya se cerró)
    presentados = datos[datos["estado"] != "Pendiente"]
    # Filtrar los datos para obtener los casos vencidos
    #   (pendientes con la fecha de fase 2 ya pasada)
    vencidos = datos[(datos["estado"] == "Pendiente") &
                     (datos["fecha_limite_fase2"] < ahora)]
    # Filtrar los datos para obtener los casos pendientes (todavía no vencidos)
    pendientes = datos[(datos["estado"] == "Pendiente") &
                       (datos["fecha_limite_fase2"] >= ahora)]

    # Contar el número de casos en cada categoría
    counts = {
        "A tiempo":   len(presentados),
        "Vencidos":   len(vencidos),
        "Pendientes": len(pendientes),
    }

    # Crear el gráfico de barras
    plt.figure(figsize=(7, 4))
    plt.bar(counts.keys(), counts.values(),
            color=["tab:green", "tab:red", "tab:orange"])

    # Configurar el título y etiquetas del gráfico
    plt.title("Casos: a tiempo vs vencidos vs pendientes")
    plt.ylabel("Cantidad")

    # Mostrar el gráfico
    plt.show()


# Llamada a la función para generar el gráfico con los datos analizados
grafico_casos_presentados_vs_vencidos_y_pendientes(cargar_datos("casos.csv"))

In [ ]:
# Función para generar la gráfica de rendimiento de abogados
def grafico_rendimiento_abogados(datos):
    """Gráfico de barras apiladas por abogado: casos a tiempo vs vencidos."""
    # Convertir las columnas de fechas a tipo datetime para facilitar las comparaciones
    datos = datos.copy()
    datos["fecha_limite_fase2"] = pd.to_datetime(datos["fecha_limite_fase2"], errors="coerce")
    ahora = datetime.now()

    # Filtrar los datos para obtener los casos presentados a tiempo
    #   (estado distinto de "Pendiente" -> ya cerrados)
    a_tiempo = datos[datos["estado"] != "Pendiente"]
    # Filtrar los datos para obtener los casos vencidos (Pendiente con fase 2 pasada)
    vencidos = datos[(datos["estado"] == "Pendiente") &
                     (datos["fecha_limite_fase2"] < ahora)]

    # Obtener la lista única de abogados en los datos
    abogados = sorted(datos["abogado"].unique())

    # Contar los casos presentados a tiempo y vencidos para cada abogado
    casos_a_tiempo = [(a_tiempo["abogado"] == ab).sum() for ab in abogados]
    casos_vencidos = [(vencidos["abogado"] == ab).sum() for ab in abogados]

    # Configurar el ancho de las barras
    ancho = 0.6

    # Crear una nueva figura y un conjunto de ejes
    fig, ax = plt.subplots(figsize=(10, 5))

    # Crear las barras apiladas para los casos presentados a tiempo y los vencidos
    ax.bar(abogados, casos_a_tiempo, ancho, label="A tiempo", color="tab:green")
    ax.bar(abogados, casos_vencidos, ancho, bottom=casos_a_tiempo,
           label="Vencidos", color="tab:red")

    # Configurar el título y etiquetas del gráfico
    ax.set_title("Rendimiento por abogado")
    ax.set_ylabel("Cantidad de casos")
    ax.legend()

    # Rotar las etiquetas del eje X para mejor legibilidad
    plt.xticks(rotation=20)
    # Ajustar el diseño para evitar recortes
    plt.tight_layout()
    # Mostrar el gráfico
    plt.show()


# Llamada a la función para generar el gráfico con los datos analizados
grafico_rendimiento_abogados(cargar_datos("casos.csv"))

## 5. Función `main` que orquesta todo el sistema


In [ ]:
def main(archivo_csv="casos.csv"):
    """Pipeline completo del sistema de gestión (siguiendo el enunciado del C4L_6)."""
    print("=" * 60)
    print("SISTEMA DE GESTIÓN DE CASOS")
    print("=" * 60)

    # Direcciones de email para alertas e informes
    emisor   = "info@mycoollawfirm.com"
    receptor = "reporting@mycoollawfirm.com"

    # 1. Carga y análisis de datos
    datos = cargar_datos(archivo_csv)
    print(f"\nCargados {len(datos)} casos.")

    datos = calcular_fechas_limite(datos)
    pendientes = datos[datos["estado"] == "Pendiente"].copy()
    print(f"   De ellos, {len(pendientes)} están pendientes.")
    analizar_fechas(pendientes)

    # 2. Sistema de alertas — envía emails a los abogados responsables
    print("\n--- ALERTAS ---")
    revisar_fechas_y_enviar_alertas(pendientes, emisor)

    # 3. Generación y envío de informes (CSV + PDF) al departamento de reporting
    print("\n--- INFORMES ---")
    generar_y_enviar_informe_pendientes(pendientes, emisor, receptor)

    # 4. Visualización
    print("\n--- VISUALIZACIÓN ---")
    grafico_casos_presentados_vs_vencidos_y_pendientes(datos)
    grafico_rendimiento_abogados(datos)

    print("\n=== Pipeline completo ===")


main()

## Cierre del curso

¡Enhorabuena! Si has llegado hasta aquí con todo funcionando, has completado el curso **Coding for Lawyers** y has demostrado a ti mismo/a que la programación **no es exclusiva del perfil técnico**.

### Lo que te llevas

| Habilidad | Lo que puedes hacer ya |
| --- | --- |
| **Python** | Variables, tipos, fechas, métodos, condicionales, bucles, funciones, archivos. |
| **Manejo de datos** | Pandas, numpy, matplotlib para análisis y visualización. |
| **Automatización** | Pipelines completos: cargar → procesar → analizar → reportar. |
| **IA generativa** | Bases de prompt engineering aplicado al derecho. |

### Próximos pasos sugeridos

1. **Aplica lo aprendido** a una tarea concreta de tu día a día. La mejor forma de consolidar es resolviendo un problema real.
2. **Mantén una "carpeta de scripts"** con las funciones reutilizables que vayas creando.
3. **Sigue practicando** con proyectos propios y consulta la documentación oficial de Python, pandas y los proveedores de IA cuando necesites profundizar.

> Gracias por confiar en *Coding for Lawyers*. **Sigue programando.**